# 🤖 Aula 05 — Machine Learning Supervisionado

**Disciplina:** Inteligência Artificial Aplicada à Engenharia Química  
**Dataset:** Coluna de destilação etanol-água — 30 dias (8 variáveis, ~43.000 pontos)  
**Target:** `composicao_destilado` (teor de etanol no topo)  
**Features:** `T_top_C`, `T_base_C`, `P_coluna_kPa`, `F_alimentacao_kg_h`, `D_destilado_kg_h`, `B_residuo_kg_h`, `R_refluxo`

---

## Contexto

Você é o engenheiro de processos e quer **predizer a composição do destilado em tempo real** — um soft-sensor. Usando o dataset de 30 dias da coluna, vamos treinar e comparar 4 modelos de ML.

## 3.1 — Exercício Guiado: Pipeline de ML

Siga passo-a-passo as células abaixo.

### Passo 1: Importar bibliotecas

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt

# pip install xgboost (se necessário no Colab)
from xgboost import XGBRegressor

### Passo 2: Carregar o dataset + Separar X e y

In [ ]:
URL = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula04/coluna_destilacao_30dias.csv"

df = pd.read_csv(URL)
df.dropna(inplace=True)

feature_cols = ['T_top_C', 'T_base_C', 'P_coluna_kPa', 'F_alimentacao_kg_h',
                'D_destilado_kg_h', 'B_residuo_kg_h', 'R_refluxo']
target_col = 'composicao_destilado'

X = df[feature_cols]
y = df[target_col]

print(f"Features: {X.shape[1]} colunas | {X.shape[0]} linhas")
print(f"Target: {target_col} | {y.shape[0]} valores")

### Passo 3: Split treino/teste (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Treino: {X_train.shape[0]} linhas | Teste: {X_test.shape[0]} linhas")

### Passo 4: Treinar 4 modelos

In [ ]:
modelos = {
    'Regressão Linear': LinearRegression(),
    'Árvore (depth=5)': DecisionTreeRegressor(max_depth=5, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0)
}

resultados = []
for nome, model in modelos.items():
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    resultados.append({
        'Modelo': nome,
        'R² treino': r2_score(y_train, y_pred_train),
        'R² teste': r2_score(y_test, y_pred_test),
        'RMSE treino': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'RMSE teste': np.sqrt(mean_squared_error(y_test, y_pred_test)),
    })

df_result = pd.DataFrame(resultados).round(4)
print(df_result.to_markdown(index=False))

### Passo 5: Plot predicted vs real (melhor modelo)

In [ ]:
# O XGBoost geralmente é o melhor
best_model = modelos['XGBoost']
y_pred = best_model.predict(X_test)

plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.3, s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Composição real')
plt.ylabel('Composição predita')
plt.title('XGBoost — Predito vs Real (teste)')
plt.tight_layout()
plt.show()

### Passo 6: Coeficientes da Regressão Linear (interpretabilidade)

In [ ]:
lr = modelos['Regressão Linear']
for col, coef in zip(feature_cols, lr.coef_):
    print(f"  {col}: {coef:.4f}")

### ✏️ Pausa reflexiva (2 min)

Qual modelo você **acha** que vai ter o maior R² nos dados de treino? E nos de teste?

> _Escreva aqui..._

---

## 3.2 — Exercício em Grupo: Diagnóstico de Overfitting

Cada grupo testa uma variação diferente de hiperparâmetro.

| Grupo | Parâmetro | Valores |
|-------|-----------|--------|
| **A** | `max_depth` (Árvore) | 3, 5, 10 |
| **B** | `n_estimators` (Random Forest) | 10, 100, 500 |
| **C** | `max_depth` (Random Forest) | 3, 10, None |
| **D** | `learning_rate` (XGBoost) | 0.01, 0.1, 0.3 |

In [ ]:
# Grupo A exemplo: testar max_depth na Árvore
for depth in [3, 5, 10]:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    r2_train = r2_score(y_train, tree.predict(X_train))
    r2_test = r2_score(y_test, tree.predict(X_test))
    print(f"max_depth={depth}: R² treino={r2_train:.4f} | R² teste={r2_test:.4f} | gap={r2_train-r2_test:.4f}")

> **Perguntas para o grupo:**
> 1. Qual valor **minimiza a diferença** treino-teste? (menos overfitting)
> 2. Qual valor **maximiza** o R² de teste?
> 3. Os dois são iguais? Se não, qual você escolheria e por quê?

### 🧠 Desafio extra (NT)

Se você tivesse que entregar um modelo para o operador da planta usar amanhã, qual escolheria: Regressão Linear (interpretável) ou XGBoost (mais preciso)?

> _Escreva aqui..._

---

## Checklist

- [ ] Split treino/teste
- [ ] 4 modelos treinados
- [ ] Tabela comparativa
- [ ] Predicted vs real plotado
- [ ] Melhor modelo identificado
- [ ] Overfitting diagnosticado